# Demo 1：从零完成一次 TabularX 建模

本 Notebook 面向第一次使用 TabularX 的用户，覆盖公开 API 的主流程：`AutoEstimator`、参数配置、`fit()`、结果对象、报告路径和端到端推断。

本例使用 sklearn 的 Breast Cancer 二分类数据集。更完整的参数、结果契约和异常说明请参阅 [API 参考手册](../api_reference.md)。

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
from sklearn.datasets import load_breast_cancer
from tabularx import AutoEstimator


In [2]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(y.value_counts())


(569, 30)
target
1    357
0    212
Name: count, dtype: int64


## 1. 创建估计器

`AutoEstimator` 是一键建模入口。`task` 可以设置为 `"classification"`、`"regression"` 或 `"auto"`；`output_dir` 是本次运行的输出根目录；`pdf_report=True` 时会额外请求 PDF 报告。

In [3]:
ae = AutoEstimator(
    task="classification",
    output_dir="./outputs/demo1",
    pdf_report=False,
)
print(ae.param.overview())


  TabularX 参数总览

【Profile 阶段】
           text_unique_threshold =   200   #文本列判定：唯一值超过此数量 → 可能为文本列
               text_unique_ratio =   0.5   #文本列判定：唯一值占比（唯一值数/行数）超过此值 → 文本列
       int_categorical_threshold =    20   #整数列判定：唯一值数量 ≤ 此值 → 类别列
            scale_mean_threshold =   0.5   #Scale 判据：|mean| > 此值 → 需要标准化
             scale_std_threshold =   1.5   #Scale 判据：std > 此值 → 需要标准化
      high_cardinality_threshold =    50   #高基数判定：唯一值 > 此值 → high_cardinality
                  skew_threshold =   1.0   #偏斜判定：|skew| > 此值 → skewed
                 imbalance_ratio =   0.1   #类别不平衡：最小类别占比 < 此值 → imbalanced
          min_samples_regression =    10   #回归任务最少样本数（低于此值报错）
           min_samples_per_class =     2   #分类任务每类最少样本数（低于此值报错）
            enable_leakage_audit =  True   #是否执行静态数据泄漏与风险预检
         identifier_unique_ratio =  0.98   #疑似标识符特征判定阈值（唯一值比例上限）
          target_proxy_threshold =  0.98   #疑似目标代理特征评分阈值（判别力上限）
                      id_columns =    []   #用户显式指定的 ID 列名列表
              forbi

## 2. 配置参数

所有阶段的配置集中在 `ae.param`。可以直接赋值，也可以使用 `set_param()` 进行带校验的修改。这里使用少量 trial 以便教学运行较快。

In [4]:
ae.param.split.test_size = 0.2
ae.param.split.validation_size = 0.25
ae.param.split.random_state = 42
ae.param.model_search.set_param("models", ["random_forest", "xgboost"])
ae.param.model_search.set_param("n_trials", 2)
ae.param.model_search.set_param("n_jobs", 1)
ae.param.shap.set_param("max_explain_samples", 24)
ae.param.shap.set_param("top_k_features", 5)
print(ae.param.overview())


  TabularX 参数总览

【Profile 阶段】
           text_unique_threshold =   200   #文本列判定：唯一值超过此数量 → 可能为文本列
               text_unique_ratio =   0.5   #文本列判定：唯一值占比（唯一值数/行数）超过此值 → 文本列
       int_categorical_threshold =    20   #整数列判定：唯一值数量 ≤ 此值 → 类别列
            scale_mean_threshold =   0.5   #Scale 判据：|mean| > 此值 → 需要标准化
             scale_std_threshold =   1.5   #Scale 判据：std > 此值 → 需要标准化
      high_cardinality_threshold =    50   #高基数判定：唯一值 > 此值 → high_cardinality
                  skew_threshold =   1.0   #偏斜判定：|skew| > 此值 → skewed
                 imbalance_ratio =   0.1   #类别不平衡：最小类别占比 < 此值 → imbalanced
          min_samples_regression =    10   #回归任务最少样本数（低于此值报错）
           min_samples_per_class =     2   #分类任务每类最少样本数（低于此值报错）
            enable_leakage_audit =  True   #是否执行静态数据泄漏与风险预检
         identifier_unique_ratio =  0.98   #疑似标识符特征判定阈值（唯一值比例上限）
          target_proxy_threshold =  0.98   #疑似目标代理特征评分阈值（判别力上限）
                      id_columns =    []   #用户显式指定的 ID 列名列表
              forbi

## 3. 运行完整流水线

`fit(X, y)` 会依次执行数据画像、数据划分、预处理、模型搜索、解释、审计和证据交付。`sensitive_features` 是可选的敏感字段名列表；本数据集没有合适的敏感字段，因此本例不传入。

In [5]:
ae.fit(X, y)
print("运行目录：", ae.output_dir)


[I 2026-10-07 22:17:56,059] A new study created in memory with name: random_forest_binary


运行目录： C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849


## 4. 读取结果契约

拟合后，所有阶段结果都可以从 `ae.result` 访问。常用对象包括 `profile`、`split`、`preprocessed`、`searched`、`shap` 和 `fairness`。

In [9]:
result = ae.result
print("任务：", result.profile.target.task_type)
print("最佳模型：", result.searched.best_model_name)
print("主指标：", result.searched.primary_metric)
print("测试集指标：", result.searched.best_test_metrics)
print("SHAP 是否生成：", result.shap is not None)


任务： binary
最佳模型： xgboost
主指标： roc_auc
测试集指标： {'accuracy': 0.9649122807017544, 'balanced_accuracy': 0.9573412698412699, 'f1': 0.9726027397260274, 'f1_macro': 0.9619111259605746, 'roc_auc': 0.9933862433862434, 'precision': 0.9594594594594594, 'recall': 0.9861111111111112}
SHAP 是否生成： True


## 5. 查看报告和交付状态

`report_paths` 提供总报告、阶段报告和证据验收页面的路径。`evidence_.audit()` 返回机器可读的交付状态；文件生成完整不代表模型没有风险。

In [10]:
for name, path in result.report_paths.items():
    print(f"{name}: {path}")

delivery = ae.evidence_.audit()
print("交付是否完整：", delivery["complete"])
print("已生成必需项：", delivery.get("generated_required_count"))


profile: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\reports\profile.html
split: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\reports\split.html
preprocess: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\reports\preprocess.html
model_search: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\reports\model_search.html
explanation: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\reports\explanation.html
full: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\report.html
evidence: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604849\evidence\delivery.html
evidence_json: C:\Users\<user>\programs\ML26-qyr\docs\demo\outputs\demo1\runs\ac956ff9144e422c985bd6c067604

## 6. 使用保存的流水线推断

推荐加载 `pipeline.joblib`，因为它同时包含预处理器和模型，可以直接接收原始 DataFrame。`best_model.joblib` 只接收已经完成预处理的特征矩阵。只加载可信来源的 joblib 文件。

In [11]:
pipeline = joblib.load(ae.output_dir / "pipeline.joblib")
predictions = pipeline.predict(X.head(5))
print(predictions)


[0 0 0 0 0]


## 本例小结

本例覆盖了最常用的端到端接口。下一步请阅读 Demo 2，了解全部配置组、结果契约和审计对象；再阅读 Demo 3，了解证据补做、异常处理和模型持久化。